In [1]:
import numpy as np
import torch
import torch.nn as nn

from models.model import End2EndModel
from experiments.purity.purity_model import PurityModel

torch.set_printoptions(sci_mode=False)

from experiments.utils import get_device, get_dataset

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = get_device(cpu=True)

N_CONCEPT, N_CLASSES, val_loader, concept_index_to_string = get_dataset("three", return_points=False, return_orig=False, return_path=False, shuffle=True)

XtoCtoY = End2EndModel(num_classes=6, num_concepts=N_CONCEPT, use_sigmoid=False, use_modulelist=False)
XtoCtoY.load_state_dict(torch.load("./../saves/converted/independent-three-seed-950.pth"))
XtoC = XtoCtoY.x_to_c_model
XtoC.to(device)
XtoC.eval()

purity_model = PurityModel(num_concepts=XtoC.out_fc.out_features)
purity_model.load_state_dict(torch.load("./../saves/independent-three-seed-950_OIS_ground.pth"))
purity_model.to(device)
purity_model.eval()

print("models loaded")

Device: cpu NVIDIA GeForce RTX 3050 Ti Laptop GPU
models loaded


# True concept values

In [3]:
imageId = 1

images, labels, concepts = next(iter(val_loader))
image = images[imageId].to(device)

print("True concepts:", concepts[imageId])

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(concepts[imageId]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},")

True concepts: tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
id: 2, concept name: 2H,
id: 15, concept name: 5S,
id: 17, concept name: 6D,


# X to C model concept predictions

In [4]:
image = image.unsqueeze(0)

prediction = XtoC(image)
pred_concepts_readable = torch.nn.Sigmoid()(prediction)

print(pred_concepts_readable)

# print concept as string if prediction is >= 0.5
for idx, i in enumerate(pred_concepts_readable[0]):
    if i.item() >= 0.5:
        print(f"id: {idx},", f"concept name: {concept_index_to_string(idx)},", f"sig value: {i.item()}")

tensor([[    0.0000,     0.0000,     1.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0001,     0.0000,     1.0000,     0.0000,     1.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
             0.0000,     0.0000,     0.0000,     0.0000]],
       grad_fn=<SigmoidBackward0>)
id: 2, concept name: 2H, sig value: 1.0
id: 15, concept name: 5S, sig value: 1.0
id: 17, concept name: 6D, sig value: 0.9999983310699463


# MLP concept predictions from single concepts

In [5]:
one_hot = nn.functional.one_hot(torch.arange(0, concepts.size(1)), num_classes=concepts.size(1)).to(device)
concepts_in = []
for i in range(prediction.size(0)):
    concepts_in.append(torch.mul(prediction[i], one_hot))

# x_to_c_model concept predictions are turned into one hot like vectors
# each concept prediction is in an array of size (0, num_concepts) where all but that concept has the value of 0
# the new batch size for purity_model training is original batch size * num_concepts
concepts_in = torch.cat(concepts_in)

# repeate concepts to use as lables for concepts_in
target_concepts = torch.repeat_interleave(prediction.detach(), concepts_in.size(1), dim=0)

outputs = torch.nn.Sigmoid()(torch.cat(purity_model(concepts_in), dim=1))
print(outputs[0])
outputs_readable = torch.nn.Sigmoid()(outputs)

for idx, i in enumerate(outputs_readable):
    print(f"input concept: {concept_index_to_string(idx)}")
    print(i)

    # print concept as string if prediction is >= 0.5
    for idy, j in enumerate(i):
        if j.item() >= 0.5:
            print(f"id: {idy},", f"concept name: {concept_index_to_string(idy)},", f"sig value: {j.item()}")

tensor([0.4295, 0.4899, 0.4974, 0.4960, 0.5112, 0.4690, 0.4799, 0.4911, 0.4990,
        0.5383, 0.5384, 0.4754, 0.5084, 0.5232, 0.4893, 0.5196, 0.5080, 0.4373,
        0.5077, 0.5559, 0.4063, 0.4360, 0.5628, 0.5307, 0.5279, 0.4841, 0.5119,
        0.5297, 0.5639, 0.5006, 0.5043, 0.4935, 0.5021, 0.4884, 0.5594, 0.4913,
        0.4621, 0.5487, 0.4978, 0.4571, 0.5025, 0.5358, 0.5644, 0.4350, 0.4877,
        0.5339, 0.5389, 0.4867, 0.5233, 0.4069, 0.5536, 0.5175],
       grad_fn=<SelectBackward0>)
input concept: 2C
tensor([0.6057, 0.6201, 0.6218, 0.6215, 0.6251, 0.6152, 0.6177, 0.6204, 0.6222,
        0.6314, 0.6314, 0.6167, 0.6244, 0.6279, 0.6199, 0.6270, 0.6243, 0.6076,
        0.6243, 0.6355, 0.6002, 0.6073, 0.6371, 0.6297, 0.6290, 0.6187, 0.6253,
        0.6294, 0.6374, 0.6226, 0.6235, 0.6209, 0.6229, 0.6197, 0.6363, 0.6204,
        0.6135, 0.6338, 0.6219, 0.6123, 0.6231, 0.6308, 0.6375, 0.6071, 0.6196,
        0.6304, 0.6316, 0.6193, 0.6279, 0.6003, 0.6350, 0.6266],
       grad_fn=<Un